In [1]:
from collections import Counter

import osmium
import pandas as pd

from isochrone.download import RAW_DATA_DIR, osm_timestamp

OSM_PATH = RAW_DATA_DIR / "oberbayern_2026-11-05.osm.pbf"
print("data up to", osm_timestamp(OSM_PATH))

highway = Counter()
other = {key: Counter() for key in ["foot", "access", "sidewalk", "motorroad", "area", "indoor"]}
ways = osmium.FileProcessor(OSM_PATH, osmium.osm.WAY).with_filter(
    osmium.filter.KeyFilter("highway")
)
for way in ways:
    highway[way.tags["highway"]] += 1
    for key, counter in other.items():
        if key in way.tags:
            counter[way.tags[key]] += 1

for key, counter in other.items():
    print(key, counter.most_common(6))
print(sum(highway.values()), "ways with a highway tag,", len(highway), "values")
pd.Series(highway).sort_values(ascending=False)


data up to 2026-11-03T21:24:23Z
foot [('yes', 33881), ('designated', 32707), ('use_sidepath', 12263), ('no', 4370), ('permissive', 1383), ('private', 746)]
access [('private', 103023), ('no', 5082), ('permissive', 3879), ('yes', 3816), ('customers', 3431), ('destination', 2033)]
sidewalk [('no', 32554), ('both', 18356), ('right', 7358), ('separate', 6563), ('left', 5954), ('yes', 40)]
motorroad [('yes', 2140), ('no', 653)]
area [('yes', 3427), ('no', 40)]
indoor [('yes', 5896), ('no', 101), ('room', 47), ('corridor', 10), ('footway', 3), ('highway', 3)]
984636 ways with a highway tag, 41 values


service              285590
track                198096
footway              159335
path                 105518
residential          100367
unclassified          34269
secondary             23364
tertiary              22099
steps                 16516
primary               10158
living_street          6135
cycleway               4759
motorway               3740
bus_stop               2173
motorway_link          2139
pedestrian             1966
trunk                  1480
primary_link           1119
trunk_link             1095
corridor               1084
secondary_link          841
proposed                824
construction            553
platform                519
tertiary_link           302
bridleway               197
elevator                 96
raceway                  80
rest_area                70
via_ferrata              67
services                 29
busway                   15
road                     13
yes                      13
ladder                    5
disused             

In [2]:
import time

import numpy as np
from scipy.sparse.csgraph import connected_components

from isochrone.osm import build_walking_graph

start = time.perf_counter()
graph = build_walking_graph(OSM_PATH)
print(f"built in {time.perf_counter() - start:.0f} s")

n_nodes = len(graph.node_ids)
print(n_nodes, "nodes,", graph.lengths.nnz // 2, "edges")
print(f"total length {graph.lengths.sum() / 2 / 1000:.0f} km")
print("edge length (m), percentiles 50, 90, 99, 100:")
print(np.percentile(graph.lengths.data, [50, 90, 99, 100]).round(1))

n_components, component = connected_components(graph.lengths, directed=False)
sizes = np.sort(np.bincount(component))[::-1]
print(n_components, f"components; the largest has {sizes[0] / n_nodes:.1%} of the nodes")
print("sizes of the 10 largest:", sizes[:10])
print("components with at most 10 nodes:", (sizes <= 10).sum())


built in 12 s
5156527 nodes, 5450977 edges
total length 130456 km
edge length (m), percentiles 50, 90, 99, 100:
[  15.    50.4  148.6 1845.6]
6854 components; the largest has 98.9% of the nodes
sizes of the 10 largest: [5098093     982     731     643     584     408     311     308     292
     261]
components with at most 10 nodes: 5600


In [3]:
import shapely

from isochrone.osm import read_poly
from isochrone.transfers import load_platforms
from isochrone.walking import build_grid, to_projected

FEED_PATH = RAW_DATA_DIR / "mvv_gtfs_2026-11-01.zip"
POLY_PATH = RAW_DATA_DIR / "oberbayern_2026-11-05.poly"

platforms = load_platforms(FEED_PATH)
grid = build_grid(platforms)

area = read_poly(POLY_PATH)
area_xy = shapely.transform(area, lambda lonlat: np.column_stack(to_projected(*lonlat.T)))
shapely.prepare(area_xy)
print(f"polygon: {len(area.exterior.coords)} vertices, {area_xy.area / 1e6:.0f} km2")

inside = shapely.contains_xy(area_xy, grid[:, 0], grid[:, 1])
print(f"grid cells inside {inside.sum()}, outside {(~inside).sum()} ({(~inside).mean():.1%})")
platform_inside = shapely.contains_xy(area_xy, platforms["x"], platforms["y"])
print("platforms outside", (~platform_inside).sum())
grid = grid[inside]


polygon: 493 vertices, 17960 km2
grid cells inside 724685, outside 201056 (21.7%)
platforms outside 2824


In [4]:
from isochrone.osm import walking_distances

platform_xy = platforms[["x", "y"]].to_numpy()
start = time.perf_counter()
pairs = walking_distances(graph, platform_xy, grid)
print(f"computed in {time.perf_counter() - start:.0f} s: {len(pairs)} platform-cell pairs")

pairs["air_m"] = np.hypot(*(platform_xy[pairs["source"]] - grid[pairs["target"]]).T)
far = pairs[pairs["air_m"] > 300]
ratio = far["distance_m"] / far["air_m"]
print("network / straight line (pairs over 300 m), percentiles 10, 25, 50, 75, 90:")
print(np.percentile(ratio, [10, 25, 50, 75, 90]).round(2))

has_platform = np.zeros(len(grid), dtype=bool)
has_platform[pairs["target"]] = True
print(f"cells with a platform within 1300 m on the network: {has_platform.sum()}"
      f" of {len(grid)} ({has_platform.mean():.1%})")
print("platforms per cell, percentiles 10, 50, 90:",
      np.percentile(pairs.groupby("target").size(), [10, 50, 90]))


computed in 173 s: 4205608 platform-cell pairs
network / straight line (pairs over 300 m), percentiles 10, 25, 50, 75, 90:
[1.13 1.2  1.31 1.46 1.69]
cells with a platform within 1300 m on the network: 612691 of 724685 (84.5%)
platforms per cell, percentiles 10, 50, 90: [ 2.  4. 14.]


In [5]:
import datetime as dt

import pandas as pd

from isochrone.connections import build_connections
from isochrone.csa import build_network, latest_departures
from isochrone.gtfs import read_gtfs_table
from isochrone.maps import band_polygons, isochrone_map
from isochrone.osm import find_nearby_platforms_on_network
from isochrone.transfers import build_transfers
from isochrone.walking import latest_departures_from_points, walking_time_s

connections = build_connections(FEED_PATH, dt.date(2026, 12, 9))
network = build_network(connections, build_transfers(FEED_PATH))
arrival = 9 * 3600

stops = read_gtfs_table(FEED_PATH, "stops", usecols=["stop_id", "stop_lat", "stop_lon"])
garching = stops.set_index("stop_id").loc["de:09184:460"]
garching_lonlat = (float(garching["stop_lon"]), float(garching["stop_lat"]))
dest_xy = np.column_stack(to_projected(*garching_lonlat))

egress = find_nearby_platforms_on_network(graph, dest_xy, platforms)
targets = dict(zip(egress["stop_id"], (arrival - egress["walk_s"]).tolist()))
departures = latest_departures(network, targets)

start = time.perf_counter()
access = find_nearby_platforms_on_network(graph, grid, platforms)
print(f"access pairs: {len(access)} in {time.perf_counter() - start:.0f} s")

direct = walking_distances(graph, dest_xy, grid, max_distance_m=7200.0)
direct_walk_s = np.full(len(grid), np.nan)
direct_walk_s[direct["target"]] = walking_time_s(direct["distance_m"], detour_factor=1.0)

latest_osm = latest_departures_from_points(access, departures, direct_walk_s, arrival)
minutes_osm = (arrival - latest_osm) / 60
print(f"cells reaching Garching within 2 h: {np.isfinite(minutes_osm).sum()} of {len(grid)}")

bands = band_polygons(grid, minutes_osm)
display(bands[["low_min", "high_min", "n_cells"]])
map_path = RAW_DATA_DIR.parent / "maps" / "garching_2026-12-09_0900_osm.html"
isochrone_map(bands, garching_lonlat).save(map_path)
print(map_path)


access pairs: 4205608 in 171 s
cells reaching Garching within 2 h: 226230 of 724685


,low_min,high_min,n_cells
0,0.0,5.0,16
1,5.0,10.0,81
2,10.0,15.0,213
3,15.0,20.0,430
4,20.0,25.0,771
5,25.0,30.0,1334
6,30.0,35.0,2224
7,35.0,40.0,3240
8,40.0,45.0,3793
9,45.0,50.0,4912


/Users/francescocodeca/Desktop/project_maps/data/maps/garching_2026-12-09_0900_osm.html


In [6]:
from isochrone.maps import DEFAULT_BANDS_MIN
from isochrone.walking import find_nearby_platforms

egress_air = find_nearby_platforms(dest_xy, platforms)
targets_air = dict(zip(egress_air["stop_id"], (arrival - egress_air["walk_s"]).tolist()))
departures_air = latest_departures(network, targets_air)
access_air = find_nearby_platforms(grid, platforms)
direct_air_s = walking_time_s(np.hypot(*(grid - dest_xy).T))
latest_air = latest_departures_from_points(access_air, departures_air, direct_air_s, arrival)
minutes_air = (arrival - latest_air) / 60

reach_air, reach_osm = np.isfinite(minutes_air), np.isfinite(minutes_osm)
both = reach_air & reach_osm
print(f"reachable within 2 h: both {both.sum()}, only straight line {(reach_air & ~both).sum()},"
      f" only OSM {(reach_osm & ~both).sum()}")

diff = minutes_osm[both] - minutes_air[both]
print("OSM minus straight line (min), percentiles 5, 25, 50, 75, 95:")
print(np.percentile(diff, [5, 25, 50, 75, 95]).round(1))
print(f"OSM slower by more than 5 min: {(diff > 5).mean():.1%},"
      f" faster by more than 5 min: {(diff < -5).mean():.1%}")

limits = np.asarray(DEFAULT_BANDS_MIN)
band_air = np.maximum(np.searchsorted(limits, minutes_air[both]), 1)
band_osm = np.maximum(np.searchsorted(limits, minutes_osm[both]), 1)
print("band shift (OSM band minus straight-line band, clipped to ±2), share of cells:")
shift = pd.Series(np.clip(band_osm - band_air, -2, 2))
print(shift.value_counts(normalize=True).sort_index().round(3))


reachable within 2 h: both 222687, only straight line 37249, only OSM 3543
OSM minus straight line (min), percentiles 5, 25, 50, 75, 95:
[-2.  -0.3  0.7  2.4  7.3]
OSM slower by more than 5 min: 9.6%, faster by more than 5 min: 0.6%
band shift (OSM band minus straight-line band, clipped to ±2), share of cells:
-2    0.001
-1    0.026
 0    0.829
 1    0.129
 2    0.015
Name: proportion, dtype: float64
